# 01 - Preparación de Datos

Este notebook cubre las fases CRISP-DM de entendimiento del negocio, entendimiento de datos y preparación. La regla metodológica central es NO usar información de test ni datos sintéticos para seleccionar, escalar o validar modelos.

In [ ]:
from pathlib import Path
import json
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.f1_pipeline import (
    CATEGORICAL_FEATURES,
    DEFAULT_SELECTED_FEATURES,
    FeatureEngineer,
    LeakageColumnDropper,
    ColumnSelector,
    Winsorizer,
    smotenc_feature_indices,
)

RANDOM_STATE = 42
DATA_PATH = ROOT / 'f1_final_dataset.csv'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 80)
print(f'Root: {ROOT}')

## Fase 1: Entendimiento del Negocio

El objetivo predictivo es estimar si un piloto terminará una carrera (`finished = 1`) usando solo variables disponibles antes o al inicio de la carrera. Por eso `laps` se elimina: se conoce después del evento y produciría leakage directo.

Criterio de éxito: comparar al menos 4 modelos supervisados y 3 ensambles, seleccionar con validación cruzada sobre el 70% original, aplicar ANOVA/Tukey, optimizar los 3 mejores y desplegar un pipeline que acepte datos crudos pre-carrera.

## Fase 2: Entendimiento de los Datos

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset original: {df.shape[0]:,} filas x {df.shape[1]} columnas')
display(df.head())

target_counts = df['finished'].value_counts().sort_index()
target_pct = df['finished'].value_counts(normalize=True).sort_index().mul(100)
display(pd.DataFrame({'conteo': target_counts, 'porcentaje': target_pct.round(2)}))

In [ ]:
quality = {
    'finished_binario': bool(df['finished'].isin([0, 1]).all()),
    'grid_en_rango_0_34': bool(df['grid'].between(0, 34).all()),
    'driver_age_en_rango_17_60': bool(df['driver_age'].between(17, 60).all()),
    'sin_nulos': bool(df.isna().sum().sum() == 0),
    'laps_presente_pero_excluida_por_leakage': 'laps' in df.columns,
}
display(pd.Series(quality, name='cumple'))

desc = df.describe().T
desc['missing'] = df.isna().sum()
desc['missing_pct'] = df.isna().mean().mul(100)
display(desc[['count', 'missing', 'missing_pct', 'mean', 'std', 'min', '50%', 'max']].round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', ax=ax, color=['#c94c4c', '#2f9e44'])
ax.set_title('Distribución de finished')
ax.set_xlabel('finished')
ax.set_ylabel('registros')
ax.set_xticklabels(['0: no terminó', '1: terminó'], rotation=0)
for idx, value in enumerate(target_counts):
    ax.text(idx, value, f'{value:,}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'distribucion_finished.png', dpi=140, bbox_inches='tight')
plt.show()

## Fase 3: Preparación de Datos

La división train/test se hace ANTES de cualquier selección, winsorización, escalado o balanceo. Las decisiones de selección de variables se justifican con diagnósticos calculados solo sobre el 70% de entrenamiento original.

In [ ]:
df_model = df.drop(columns=['laps'])
X = df_model.drop(columns=['finished'])
y = df_model['finished']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

print('Split estratificado:')
print(f'Train original: {X_train_raw.shape[0]:,} filas')
print(y_train.value_counts().sort_index().to_dict())
print(f'Test original: {X_test_raw.shape[0]:,} filas')
print(y_test.value_counts().sort_index().to_dict())

In [ ]:
# Diagnóstico de correlación y MI SOLO con train original.
train_diag = X_train_raw.copy()
train_diag['finished'] = y_train.values
corr_target = train_diag.corr(numeric_only=True)['finished'].drop('finished').abs().sort_values(ascending=False)

fe_diag = FeatureEngineer().fit_transform(X_train_raw)
mi_X = fe_diag.drop(columns=[c for c in fe_diag.columns if c not in DEFAULT_SELECTED_FEATURES], errors='ignore')
mi_scores = mutual_info_classif(mi_X, y_train, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({'feature': mi_X.columns, 'mutual_information': mi_scores}).sort_values('mutual_information', ascending=False)

display(pd.DataFrame({'abs_corr_finished_train': corr_target}).head(20).round(4))
display(mi_df.round(5))

In [ ]:
selected_features = DEFAULT_SELECTED_FEATURES
removed_features = sorted(set(X_train_raw.columns).union({'experience_ratio', 'grid_above_avg', 'avg_finish_rate'}) - set(selected_features))

print(f'Features seleccionadas ({len(selected_features)}):')
print(selected_features)
print('\nFeatures excluidas por leakage/redundancia/baja utilidad diagnóstica:')
print(removed_features)
print('\nCategóricas para SMOTENC:')
print(CATEGORICAL_FEATURES)

In [ ]:
prep_no_sampler = Pipeline([
    ('drop_leakage', LeakageColumnDropper()),
    ('feature_engineering', FeatureEngineer()),
    ('winsorizer', Winsorizer()),
    ('selector', ColumnSelector(selected_features)),
    ('scaler', StandardScaler()),
])

prep_for_smote = Pipeline([
    ('drop_leakage', LeakageColumnDropper()),
    ('feature_engineering', FeatureEngineer()),
    ('winsorizer', Winsorizer()),
    ('selector', ColumnSelector(selected_features)),
])

X_train_selected = prep_for_smote.fit_transform(X_train_raw, y_train)
X_test_selected = prep_for_smote.transform(X_test_raw)

sampler = SMOTENC(
    categorical_features=smotenc_feature_indices(selected_features),
    random_state=RANDOM_STATE,
    k_neighbors=5,
)
X_train_balanced, y_train_balanced = sampler.fit_resample(X_train_selected, y_train)

scaler = StandardScaler()
X_train_balanced_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_balanced), columns=selected_features
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_selected), columns=selected_features
)

train_balanced = X_train_balanced_scaled.copy()
train_balanced['finished'] = y_train_balanced.to_numpy()
test_unbalanced = X_test_scaled.copy()
test_unbalanced['finished'] = y_test.to_numpy()

train_original = X_train_raw.copy()
train_original['finished'] = y_train.to_numpy()
test_original = X_test_raw.copy()
test_original['finished'] = y_test.to_numpy()

train_balanced.to_csv(OUTPUT_DIR / 'train_balanced.csv', index=False)
test_unbalanced.to_csv(OUTPUT_DIR / 'test_unbalanced.csv', index=False)
train_original.to_csv(OUTPUT_DIR / 'train_original.csv', index=False)
test_original.to_csv(OUTPUT_DIR / 'test_unbalanced_raw.csv', index=False)

with open(OUTPUT_DIR / 'preprocessing_pipe.pkl', 'wb') as f:
    pickle.dump(prep_no_sampler, f)

schema = {
    'target': 'finished',
    'leakage_columns_removed': ['laps'],
    'selected_features': selected_features,
    'categorical_features_for_smotenc': CATEGORICAL_FEATURES,
    'smotenc_categorical_indices': smotenc_feature_indices(selected_features),
    'cv_rule': 'CV se ejecuta sobre train_original.csv con SMOTENC dentro de cada fold, no sobre train_balanced.csv.',
}
with open(OUTPUT_DIR / 'feature_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print('Artefactos guardados:')
print(f"train_original.csv: {train_original.shape}")
print(f"train_balanced.csv: {train_balanced.shape}; clases {pd.Series(y_train_balanced).value_counts().sort_index().to_dict()}")
print(f"test_unbalanced.csv: {test_unbalanced.shape}; clases {y_test.value_counts().sort_index().to_dict()}")
print('preprocessing_pipe.pkl y feature_schema.json actualizados')

## Resumen metodológico de preparación

- `laps` se eliminó por ser información posterior a la carrera.
- El split 70/30 se hizo antes de selección, winsorización, escalado y SMOTE.
- `train_balanced.csv` documenta el balanceo exigido sobre el 70%, pero NO se usa para validar modelos.
- La validación cruzada correcta ocurre en el notebook 02 sobre `train_original.csv`, con `SMOTENC` dentro del pipeline de cada fold.